In [1]:
import gymnasium as gym
from gymnasium import spaces
from gymnasium.utils.env_checker import check_env

In [2]:
class Reach10Env(gym.Env):
    def __init__(self):
        super().__init__()

        self.observation_space = spaces.Discrete(21)

        self.action_space = spaces.Discrete(2)

        self.steps = 0
        self.max_steps = 30
        self.position = 0

    def reset(self, *, seed = None, options = None):
        super().reset(seed=seed, options=options)
        self.steps = 0
        if options and "position" in options:
            self.position = options['position']
        else:
            self.position = self.np_random.integers(0, 21)
        return self.position, {}

    def step(self, action):
        self.steps += 1
        if action == 0:
            self.position -= 1
        elif action == 1:
            self.position += 1
        self.position = max(0, min(20, self.position))
        terminated = self.position == 0 or self.position == 20
        truncated = self.steps == self.max_steps
        if terminated:
            reward = 10
        else:
            reward = -1
        return self.position, reward, terminated, truncated, {}

In [3]:
env = Reach10Env()

check_env(env=env)

/home/hashterx/hackatonvenv/lib/python3.13/site-packages/gymnasium/utils/env_checker.py:434: UserWarning: WARN: Not able to test alternative render modes due to the environment not having a spec. Try instantiating the environment through `gymnasium.make`
  logger.warn(


In [4]:
from stable_baselines3 import PPO

In [5]:
agent = PPO('MlpPolicy', env=env, verbose=1)

agent.learn(total_timesteps=10_000)

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/home/hashterx/hackatonvenv/lib/python3.13/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 21.3     |
|    ep_rew_mean     | -16.4    |
| time/              |          |
|    fps             | 1274     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 19.4        |
|    ep_rew_mean          | -12.9       |
| time/                   |             |
|    fps                  | 1056        |
|    iterations           | 2           |
|    time_elapsed         | 3           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.014893513 |
|    clip_fraction        | 0.11        |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.687      |
|    explained_variance   | 0.00174     |
|    learning_rate        | 0.

In [6]:
obs, info = env.reset(options={'position': 10})
step = 0
while True:
    action, _ = agent.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    step += 1
    print(
        f"Step       : {step}, "
        f"Action     : {action}, "
        f"Observation: {obs}"
    )
    if terminated or truncated:
        break

Step       : 1, Action     : 1, Observation: 11
Step       : 2, Action     : 1, Observation: 12
Step       : 3, Action     : 1, Observation: 13
Step       : 4, Action     : 1, Observation: 14
Step       : 5, Action     : 1, Observation: 15
Step       : 6, Action     : 1, Observation: 16
Step       : 7, Action     : 1, Observation: 17
Step       : 8, Action     : 1, Observation: 18
Step       : 9, Action     : 1, Observation: 19
Step       : 10, Action     : 1, Observation: 20
